# Prophet Time Series Forecasting

This notebook continues from the ARIMA/SARIMA and Exponential Smoothing notebooks.

The goal is to test Prophet as another forecasting approach using the same daily `unit_sales` time series.

Prophet is designed to model time series using trend, seasonality, and optional external regressors. In this notebook, I keep the setup consistent with the previous notebooks so the results can be compared fairly.

I will:

- load the same prepared feature dataset
- keep the same chronological train/test split
- train a Prophet model with weekly seasonality
- optionally test Prophet with external variables
- compare the results with SARIMA, SARIMAX, and Holt-Winters

## Step 1 — Install and import libraries

If Prophet is not installed in your environment, run the installation line below once.

In VS Code, it is usually better to install packages in the terminal connected to your project environment.

In [ ]:
# Uncomment this line only if Prophet is not installed in your environment
# !pip install prophet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from prophet import Prophet
    print("Prophet imported successfully.")
except ImportError:
    raise ImportError("Prophet is not installed. Run: pip install prophet")

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

## Step 2 — Load the feature dataset

The dataset comes from the feature engineering notebook and contains the cleaned daily sales series together with calendar, oil, holiday, lag, and rolling features.

In [ ]:
features_path = Path("../outputs/timeseries_with_features.csv")

print("Path:", features_path.resolve())
print("Exists:", features_path.exists())

df = pd.read_csv(features_path, parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())

display(df.head())

## Step 3 — Prepare the time series

Prophet expects two main columns:

- `ds`: the date column
- `y`: the target variable to forecast

Before creating that format, I first keep the same `unit_sales` time series used in the previous notebooks.

In [ ]:
ts = df[["date", "unit_sales"]].copy()

ts = (
    ts
    .set_index("date")
    .asfreq("D")
)

print("Time series frequency:", ts.index.freq)
print("Missing values:")
print(ts.isna().sum())

display(ts.head())

The target variable `unit_sales` was selected and indexed by date.

A daily frequency was enforced using `asfreq("D")`. The validation confirms that the series has a regular daily frequency and contains no missing values, so it can be used directly for Prophet modeling.

## Step 4 — Visualize the sales series

Before fitting Prophet, I inspect the series again to understand the trend, seasonality, and noise.

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(ts.index, ts["unit_sales"])
plt.title("Daily Unit Sales Over Time")
plt.xlabel("Date")
plt.ylabel("Unit sales")
plt.grid(True)
plt.show()

The series shows a lot of day-to-day variation, with frequent spikes and drops in sales.

There is a visible weekly pattern, although it is not perfectly regular. There is no clear long-term trend, but the data is quite noisy, which is typical for daily retail sales.

Because of this, Prophet will mainly be tested for its ability to capture weekly seasonality.

## Step 5 — Train/test split

The split must respect chronological order.

In [ ]:
test_size = int(len(ts) * 0.2)

train = ts.iloc[:-test_size].copy()
test = ts.iloc[-test_size:].copy()

print("Training set:", train.index.min(), "to", train.index.max(), "| rows:", len(train))
print("Test set:", test.index.min(), "to", test.index.max(), "| rows:", len(test))

I use the first 80% of the data for training and the last 20% for testing. This way, the model is evaluated on future data it has not seen before.

The test set covers about 12 weeks, which is useful to see how well the model captures the weekly pattern.

## Step 6 — Create evaluation function

I use the same metrics as in the previous notebooks so the results are directly comparable.

In [ ]:
def evaluate_forecast(model_name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"{model_name}")
    print(f"MAE: {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R²: {r2:.4f}
")

    return {
        "model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

## Step 7 — Prepare Prophet format

Prophet requires the columns to be named `ds` and `y`.

In [ ]:
prophet_train = train.reset_index().rename(columns={"date": "ds", "unit_sales": "y"})
prophet_test = test.reset_index().rename(columns={"date": "ds", "unit_sales": "y"})

print("Prophet training data:")
display(prophet_train.head())

print("Prophet test data:")
display(prophet_test.head())

## Step 8 — Baseline Prophet model with weekly seasonality

The first Prophet model uses weekly seasonality because the previous notebooks showed that weekly patterns are important for this time series.

Yearly seasonality is disabled because the dataset covers only a little more than one year, which is not enough to reliably learn a yearly pattern.

In [ ]:
prophet_model = Prophet(
    growth="linear",
    weekly_seasonality=False,
    daily_seasonality=False,
    yearly_seasonality=False,
    seasonality_mode="additive"
)

prophet_model.add_seasonality(
    name="weekly",
    period=7,
    fourier_order=3
)

prophet_model.fit(prophet_train)

future = prophet_test[["ds"]].copy()
prophet_forecast = prophet_model.predict(future)

prophet_predictions = pd.Series(
    prophet_forecast["yhat"].values,
    index=test.index
).clip(lower=0)

prophet_results = evaluate_forecast(
    "Prophet Weekly Seasonality",
    test["unit_sales"],
    prophet_predictions
)

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(test.index, test["unit_sales"], label="Actual test data")
plt.plot(test.index, prophet_predictions, label="Prophet forecast", linestyle="--")

plt.title("Prophet Forecast vs Actual Sales")
plt.xlabel("Date")
plt.ylabel("Unit sales")
plt.legend()
plt.grid(True)
plt.show()

The Prophet forecast should be checked visually against the actual values.

If the model captures the weekly pattern, the forecast should show repeated ups and downs. However, like the other classical models, it may still produce smoother predictions than the actual sales and miss sharp spikes.

## Step 9 — Inspect Prophet components

Prophet separates the forecast into components such as trend and seasonality.

This helps us understand what the model is learning.

In [ ]:
fig = prophet_model.plot_components(prophet_forecast)
plt.show()

The components plot helps confirm whether Prophet is mainly relying on weekly seasonality or whether it is also learning a meaningful trend.

For this dataset, I expect the weekly component to be more important than the trend, because the original series does not show a clear long-term upward or downward movement.

## Step 10 — Small Prophet parameter search

Prophet has parameters that control how flexible the trend and seasonality are.

Here, I test a small grid. The goal is not to over-tune the model, but to check whether a slightly different configuration improves the forecast.

In [ ]:
param_grid = [
    {"changepoint_prior_scale": 0.01, "seasonality_prior_scale": 1.0, "fourier_order": 3},
    {"changepoint_prior_scale": 0.05, "seasonality_prior_scale": 1.0, "fourier_order": 3},
    {"changepoint_prior_scale": 0.10, "seasonality_prior_scale": 1.0, "fourier_order": 3},
    {"changepoint_prior_scale": 0.05, "seasonality_prior_scale": 5.0, "fourier_order": 3},
    {"changepoint_prior_scale": 0.05, "seasonality_prior_scale": 10.0, "fourier_order": 5},
]

prophet_grid_results = []
prophet_grid_predictions = {}

for params in param_grid:
    model = Prophet(
        growth="linear",
        weekly_seasonality=False,
        daily_seasonality=False,
        yearly_seasonality=False,
        seasonality_mode="additive",
        changepoint_prior_scale=params["changepoint_prior_scale"],
        seasonality_prior_scale=params["seasonality_prior_scale"]
    )

    model.add_seasonality(
        name="weekly",
        period=7,
        fourier_order=params["fourier_order"]
    )

    model.fit(prophet_train)
    forecast = model.predict(future)

    preds = pd.Series(forecast["yhat"].values, index=test.index).clip(lower=0)

    mae = mean_absolute_error(test["unit_sales"], preds)
    rmse = np.sqrt(mean_squared_error(test["unit_sales"], preds))
    r2 = r2_score(test["unit_sales"], preds)

    result = {
        "changepoint_prior_scale": params["changepoint_prior_scale"],
        "seasonality_prior_scale": params["seasonality_prior_scale"],
        "fourier_order": params["fourier_order"],
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

    prophet_grid_results.append(result)
    prophet_grid_predictions[str(params)] = preds

prophet_grid_df = pd.DataFrame(prophet_grid_results).sort_values("MAE").reset_index(drop=True)

display(prophet_grid_df)

The grid search compares a few Prophet configurations using the same test set.

The best model is selected based on MAE, because the main goal is to reduce the average forecasting error on unseen data.

## Step 11 — Train the best Prophet model

The best configuration from the small grid search is used as the final Prophet model.

In [ ]:
best_params = prophet_grid_df.iloc[0]

best_prophet_model = Prophet(
    growth="linear",
    weekly_seasonality=False,
    daily_seasonality=False,
    yearly_seasonality=False,
    seasonality_mode="additive",
    changepoint_prior_scale=best_params["changepoint_prior_scale"],
    seasonality_prior_scale=best_params["seasonality_prior_scale"]
)

best_prophet_model.add_seasonality(
    name="weekly",
    period=7,
    fourier_order=int(best_params["fourier_order"])
)

best_prophet_model.fit(prophet_train)

best_prophet_forecast = best_prophet_model.predict(future)

best_prophet_predictions = pd.Series(
    best_prophet_forecast["yhat"].values,
    index=test.index
).clip(lower=0)

best_prophet_results = evaluate_forecast(
    "Best Prophet Model",
    test["unit_sales"],
    best_prophet_predictions
)

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(test.index, test["unit_sales"], label="Actual test data")
plt.plot(test.index, best_prophet_predictions, label="Best Prophet forecast", linestyle="--")

plt.title("Best Prophet Forecast vs Actual Sales")
plt.xlabel("Date")
plt.ylabel("Unit sales")
plt.legend()
plt.grid(True)
plt.show()

The tuned Prophet model can now be compared with the baseline Prophet model and the previous classical models.

If the improvement is small, it means the default weekly seasonal structure was already capturing most of what Prophet can learn from this series.

## Step 12 — Optional Prophet model with external regressors

Prophet can also use external variables, similar to SARIMAX.

Here, I test a version using oil prices and calendar/holiday features. These variables are only valid for forecasting if they are known in advance or can be estimated for the forecast horizon.

In [ ]:
exog_cols = [
    "dcoilwtico",
    "is_holiday",
    "is_national_holiday",
    "is_regional_holiday",
    "is_local_holiday",
    "is_weekend",
    "is_month_start",
    "is_month_end"
]

available_exog_cols = [col for col in exog_cols if col in df.columns]

print("External features available:")
print(available_exog_cols)

df_model = df.set_index("date").asfreq("D")

# Keep target and external variables aligned with the same train/test dates
train_exog_df = df_model.loc[train.index, ["unit_sales"] + available_exog_cols].copy()
test_exog_df = df_model.loc[test.index, ["unit_sales"] + available_exog_cols].copy()

# Fill missing values if any
train_exog_df[available_exog_cols] = train_exog_df[available_exog_cols].ffill().bfill()
test_exog_df[available_exog_cols] = test_exog_df[available_exog_cols].ffill().bfill()

# Prophet format
prophet_train_exog = train_exog_df.reset_index().rename(columns={"date": "ds", "unit_sales": "y"})
prophet_future_exog = test_exog_df.reset_index().rename(columns={"date": "ds"})[["ds"] + available_exog_cols]

# Convert boolean columns to integers if needed
for col in available_exog_cols:
    if prophet_train_exog[col].dtype == "bool":
        prophet_train_exog[col] = prophet_train_exog[col].astype(int)
        prophet_future_exog[col] = prophet_future_exog[col].astype(int)

print("Missing values in prophet_train_exog:")
print(prophet_train_exog[["y"] + available_exog_cols].isna().sum())

print("
Missing values in prophet_future_exog:")
print(prophet_future_exog[available_exog_cols].isna().sum())

In [ ]:
prophet_exog_model = Prophet(
    growth="linear",
    weekly_seasonality=False,
    daily_seasonality=False,
    yearly_seasonality=False,
    seasonality_mode="additive",
    changepoint_prior_scale=best_params["changepoint_prior_scale"],
    seasonality_prior_scale=best_params["seasonality_prior_scale"]
)

prophet_exog_model.add_seasonality(
    name="weekly",
    period=7,
    fourier_order=int(best_params["fourier_order"])
)

for col in available_exog_cols:
    prophet_exog_model.add_regressor(col)

prophet_exog_model.fit(prophet_train_exog)

prophet_exog_forecast = prophet_exog_model.predict(prophet_future_exog)

prophet_exog_predictions = pd.Series(
    prophet_exog_forecast["yhat"].values,
    index=test.index
).clip(lower=0)

prophet_exog_results = evaluate_forecast(
    "Prophet with External Regressors",
    test["unit_sales"],
    prophet_exog_predictions
)

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(test.index, test["unit_sales"], label="Actual test data")
plt.plot(test.index, best_prophet_predictions, label="Best Prophet", linestyle="--")
plt.plot(test.index, prophet_exog_predictions, label="Prophet with external regressors", linestyle="--")

plt.title("Prophet vs Prophet with External Regressors")
plt.xlabel("Date")
plt.ylabel("Unit sales")
plt.legend()
plt.grid(True)
plt.show()

This comparison shows whether external variables help Prophet improve its forecast.

If the improvement is small, it means the model is still mainly driven by the time-based seasonal pattern. If the improvement is larger, then the external variables add useful information beyond the weekly cycle.

## Step 13 — Compare Prophet models

In [ ]:
prophet_results_df = pd.DataFrame([
    prophet_results,
    best_prophet_results,
    prophet_exog_results
]).sort_values("MAE").reset_index(drop=True)

prophet_results_df

## Step 14 — Compare with previous Week 2 models

The SARIMA, SARIMAX, and Holt-Winters results come from the previous notebooks.

This gives a single comparison table for the main Week 2 statistical forecasting models.

In [ ]:
previous_classical_results = pd.DataFrame([
    {
        "model": "SARIMAX",
        "MAE": 96.58,
        "RMSE": 139.85,
        "R2": 0.3165
    },
    {
        "model": "SARIMA",
        "MAE": 96.98,
        "RMSE": 140.64,
        "R2": 0.3088
    },
    {
        "model": "Holt-Winters Additive",
        "MAE": 99.44,
        "RMSE": 142.27,
        "R2": 0.2926
    }
])

combined_results = pd.concat(
    [previous_classical_results, prophet_results_df],
    ignore_index=True
).sort_values("MAE").reset_index(drop=True)

display(combined_results)

The final comparison shows how Prophet performs against the previous Week 2 models.

If Prophet is close to SARIMA, SARIMAX, or Holt-Winters, then it is a useful alternative. If it performs worse, it still adds value as a comparison model because it uses a different modeling approach.

The most important point is whether the model captures the weekly pattern and whether it can handle the noisy, irregular behavior of daily retail sales.

## Summary

In this notebook, I tested Prophet on the daily `unit_sales` time series.

I first trained a Prophet model with weekly seasonality, because the previous notebooks showed that weekly patterns are important for this dataset. I then tested a small parameter grid and also tried adding external regressors such as oil prices and calendar/holiday features.

Prophet provides another useful benchmark for Week 2. It is easier to configure than ARIMA/SARIMA, but its performance still depends on whether the main seasonal structure is captured well.

As with the other classical models, Prophet may still struggle with sharp spikes and sudden drops in sales. This is expected because daily retail data is noisy and contains irregular movements that are hard to capture with purely statistical models.

The next step is to compare these Week 2 statistical models with machine learning models, such as XGBoost, which can use the engineered features more directly and may capture more complex patterns.